In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, confusion_matrix
from tqdm import tqdm
import random

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [15]:
"""Dataset paths"""

TRAIN_REAL   = '/kaggle/input/datasets/lightvvcx/full-dataset/FULL_DATASET_FRAMES/train/real'
TRAIN_ATTACK = '/kaggle/input/datasets/lightvvcx/full-dataset/FULL_DATASET_FRAMES/train/attack'
TEST_REAL    = '/kaggle/input/datasets/lightvvcx/full-dataset/FULL_DATASET_FRAMES/test/real'
TEST_ATTACK  = '/kaggle/input/datasets/lightvvcx/full-dataset/FULL_DATASET_FRAMES/test/attack'

In [16]:
"""Dataset class"""

class AntispoofDataset(Dataset):
    def __init__(self, real_path, attack_path, transform=None):
        self.samples = []
        self.transform = transform

        for identity in os.listdir(real_path):
            identity_path = os.path.join(real_path, identity)
            if os.path.isdir(identity_path):
                for img_name in os.listdir(identity_path):
                    if img_name.endswith(('.jpg', '.png', '.jpeg')):
                        self.samples.append((os.path.join(identity_path, img_name), 1))

        for identity in os.listdir(attack_path):
            identity_path = os.path.join(attack_path, identity)
            if os.path.isdir(identity_path):
                for img_name in os.listdir(identity_path):
                    if img_name.endswith(('.jpg', '.png', '.jpeg')):
                        self.samples.append((os.path.join(identity_path, img_name), 0))

        real_count   = sum(1 for _, l in self.samples if l == 1)
        attack_count = sum(1 for _, l in self.samples if l == 0)
        print(f"Loaded {len(self.samples)} samples - Real: {real_count} | Attack: {attack_count}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

In [19]:
"""Data augmentation"""

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.3),
    transforms.ToTensor(),
    transforms.RandomApply([transforms.Lambda(lambda x: (x + 0.01 * torch.randn_like(x)).clamp(0, 1))], p=0.3),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])


In [20]:
"""Datasets and loaders"""

train_dataset = AntispoofDataset(TRAIN_REAL, TRAIN_ATTACK, train_transform)
test_dataset  = AntispoofDataset(TEST_REAL,  TEST_ATTACK,  test_transform)

train_loader = DataLoader(train_dataset, batch_size=64,
                          shuffle=True, num_workers=2, pin_memory=True)

test_loader  = DataLoader(test_dataset,  batch_size=64,
                          shuffle=False, num_workers=2, pin_memory=True)

Loaded 2806 samples - Real: 1333 | Attack: 1473
Loaded 1302 samples - Real: 637 | Attack: 665


## **ENHANCED MODEL with TEXTURE**

In [ ]:
"""CNN - Texture Branch Enhanced"""

class MobileNetTexture(nn.Module):
    def __init__(self):
        super().__init__()

        backbone = timm.create_model(
            'mobilenetv3_small_100',
            pretrained=True,
            features_only=True
        )

        self.backbone = backbone

        feature_info = backbone.feature_info
        mid_channels   = feature_info[2]['num_chs']
        final_channels = feature_info[-1]['num_chs']

        self.texture_branch = nn.Sequential(
            nn.Conv2d(mid_channels, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

        self.global_pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(final_channels + 64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        features = self.backbone(x)

        mid_feat   = features[2]
        final_feat = features[-1]

        texture_feat = self.texture_branch(mid_feat)

        main_feat = self.global_pool(final_feat)
        main_feat = main_feat.view(main_feat.size(0), -1)

        combined = torch.cat([main_feat, texture_feat], dim=1)

        out = self.classifier(combined)
        return out


In [22]:
"""Model setup - Texture Branch"""

model = MobileNetTexture().to(device)

criterion  = nn.BCEWithLogitsLoss()
optimizer  = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

NUM_EPOCHS = 20
best_acc   = 0

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [24]:
"""Training - Texture Branch"""

for epoch in range(NUM_EPOCHS):

    model.train()
    train_loss = 0
    correct    = 0
    total      = 0

    for images, labels in tqdm(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds       = (torch.sigmoid(outputs) > 0.5).float()
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    train_acc = 100 * correct / total

    model.eval()
    test_loss = 0
    correct   = 0
    total     = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(test_loader):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images).squeeze()
            loss    = criterion(outputs, labels)

            test_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_acc = 100 * correct / total
    auc      = roc_auc_score(all_labels, all_preds)

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Acc: {train_acc:.2f}%")
    print(f"Test Acc:  {test_acc:.2f}% | AUC: {auc:.4f}")

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), "/kaggle/working/best_model_texture-V2.pth")
        print("✓ Best model saved")

    scheduler.step()

100%|██████████| 21/21 [00:02<00:00,  8.72it/s]



Epoch 1/20
Train Acc: 93.80%
Test Acc:  86.48% | AUC: 0.9277
✓ Best model saved


100%|██████████| 21/21 [00:02<00:00,  8.87it/s]



Epoch 2/20
Train Acc: 96.04%
Test Acc:  83.03% | AUC: 0.9475


100%|██████████| 21/21 [00:02<00:00,  8.56it/s]


...


Epoch 18/20
Train Acc: 99.64%
Test Acc:  89.55% | AUC: 0.9617


100%|██████████| 21/21 [00:02<00:00,  9.07it/s]



Epoch 19/20
Train Acc: 99.22%
Test Acc:  89.55% | AUC: 0.9615


100%|██████████| 21/21 [00:02<00:00,  8.61it/s]


Epoch 20/20
Train Acc: 99.61%
Test Acc:  89.02% | AUC: 0.9618
